# mIF Pipeline Debug Notebook

This notebook is a notebook-first wrapper around the Python API in `src/mif_pipeline`.

Use it to:
- inspect resolved config paths
- dry-run the full slide plan
- run one stage at a time and rerun only the failing stage
- inspect outputs between steps

Real data paths are expected to be cluster paths, so the dry-run cells are useful locally.

This notebook assumes `mif_pipeline` is already installed in the active Python environment, for example with `pip install -e .`.

Important setup note: `setup_slide(...)` writes a starter channel map to `setup.channel_map_output`. If you want downstream steps in this notebook session to use that generated file immediately, flip `USE_GENERATED_CHANNEL_MAP = True` below and rerun the config-loading cell after the setup cell has written the file.


In [ ]:
from pathlib import Path
import json
import traceback


In [ ]:
print("Imports will use the installed mif_pipeline package from the active environment.")


In [ ]:
from mif_pipeline import (
    build_spatialdata,
    diagnose_label_overlap_instances,
    finalize_nimbus_multislide,
    load_config,
    merge_slide_ometiffs,
    qc_slide,
    run_all,
    run_instanseg,
    run_nimbus_chunked,
    run_nimbus_multislide,
    setup_slide,
)
from mif_pipeline.config import (
    get_slide_config,
    resolve_block_aliases,
    resolve_channel_entries,
    resolve_nimbus_inputs,
    resolve_nimbus_multislide_inputs,
    resolve_spatialdata_channel_entries,
)


In [ ]:
# Edit this to the config you want to run.
CONFIG_PATH = Path("~/codex/mIF-pipeline/prototyping/prototype_v2-fullslide.yaml").expanduser()
SLIDE_ID = "SLIDE-0329"
MULTISLIDE_SLIDE_IDS = ["SLIDE-0329"]
RUN_MULTISLIDE_NIMBUS = False
NIMBUS_CHUNK_INDICES = None
RUN_NIMBUS_FINALIZE = True
USE_GENERATED_CHANNEL_MAP = True
RUN_DRY = False
FORCE = False


In [ ]:
def show(value):
    print(json.dumps(value, indent=2, default=str))


def stage_call(label, func, *args, **kwargs):
    print(f"=== {label} ===")
    try:
        result = func(*args, **kwargs)
        show(result)
        return result
    except Exception as exc:
        print(f"{label} failed: {exc}")
        traceback.print_exc()
        raise


In [ ]:
print(f"Config path: {CONFIG_PATH}")
print(f"Config exists: {CONFIG_PATH.exists()}")


In [ ]:
config = load_config(CONFIG_PATH)

runtime_slide_ids = MULTISLIDE_SLIDE_IDS if RUN_MULTISLIDE_NIMBUS else [SLIDE_ID]
for target_slide_id in runtime_slide_ids:
    target_nimbus = config["slides"].setdefault(target_slide_id, {}).setdefault("nimbus", {})
    target_multislide = target_nimbus.setdefault("multislide", {})
    target_multislide["enabled"] = RUN_MULTISLIDE_NIMBUS
    if RUN_MULTISLIDE_NIMBUS:
        target_multislide["slide_ids"] = list(MULTISLIDE_SLIDE_IDS)

if USE_GENERATED_CHANNEL_MAP:
    for target_slide_id in MULTISLIDE_SLIDE_IDS:
        target_slide = get_slide_config(config, target_slide_id)
        generated_map = target_slide.get("setup", {}).get("channel_map_output")
        if not generated_map:
            raise ValueError(
                f"USE_GENERATED_CHANNEL_MAP is True but setup.channel_map_output is missing for {target_slide_id}."
            )
        config["slides"][target_slide_id]["channel_map_file"] = generated_map

slide = get_slide_config(config, SLIDE_ID)
nimbus_inputs = resolve_nimbus_inputs(config, SLIDE_ID)
if RUN_MULTISLIDE_NIMBUS:
    multislide_nimbus_inputs = resolve_nimbus_multislide_inputs(config, MULTISLIDE_SLIDE_IDS)
else:
    multislide_nimbus_inputs = {
        "slide_ids": [],
        "fov_paths": [],
        "output_dir": None,
        "nimbus_channels": [],
    }
instanseg_aliases = resolve_block_aliases(
    config,
    SLIDE_ID,
    slide.get("instanseg", {}) or {},
    block_name="InstanSeg block",
    require_selection=True,
)
full_aliases = resolve_block_aliases(
    config,
    SLIDE_ID,
    slide.get("full_merge", {}) or {},
    block_name="Merge block",
    require_selection=False,
)
instanseg_entries = resolve_channel_entries(config, SLIDE_ID, instanseg_aliases)
full_entries = resolve_channel_entries(config, SLIDE_ID, full_aliases)
spatialdata_entries = resolve_spatialdata_channel_entries(config, SLIDE_ID)
cell_mask_path = Path(slide["mask_export"]["mask_dir"]) / f"{SLIDE_ID}{slide['mask_export']['suffix']}"
nuclear_mask_path = Path(slide["mask_export"]["mask_dir"]) / f"{SLIDE_ID}{slide['mask_export']['nuclear_suffix']}"
spatialdata_store_path = slide.get("spatialdata", {}).get("store_path")

summary = {
    "slide_id": SLIDE_ID,
    "slide_dir": slide["slide_dir"],
    "output_dir": slide["output_dir"],
    "channel_map_file": slide["channel_map_file"],
    "setup_channel_map_output": slide.get("setup", {}).get("channel_map_output"),
    "instanseg_channels": instanseg_aliases,
    "instanseg_inputs": [entry["path"] for entry in instanseg_entries],
    "full_merge_inputs": [entry["path"] for entry in full_entries],
    "nimbus_raw_paths": nimbus_inputs["raw_paths"],
    "nimbus_fov_paths": nimbus_inputs["fov_paths"],
    "multislide_slide_ids": multislide_nimbus_inputs["slide_ids"],
    "multislide_fov_paths": multislide_nimbus_inputs["fov_paths"],
    "multislide_output_dir": multislide_nimbus_inputs["output_dir"],
    "multislide_nimbus_channels": multislide_nimbus_inputs["nimbus_channels"],
    "spatialdata_aliases": [entry["alias"] for entry in spatialdata_entries],
    "spatialdata_store_path": spatialdata_store_path,
    "instanseg_image_path": slide["full_merge"]["ome_path"],
    "cell_mask_path": str(cell_mask_path),
    "nuclear_mask_path": str(nuclear_mask_path),
    "nimbus_chunk_indices": NIMBUS_CHUNK_INDICES,
}
show(summary)


In [ ]:
def import_status(module_name):
    try:
        __import__(module_name)
        return "OK"
    except Exception as exc:
        return f"MISSING: {exc}"


dependency_status = {
    "yaml": import_status("yaml"),
    "tifffile": import_status("tifffile"),
    "ome_types": import_status("ome_types"),
    "numpy": import_status("numpy"),
    "skimage": import_status("skimage"),
    "instanseg": import_status("instanseg"),
    "tiffslide": import_status("tiffslide"),
    "anndata": import_status("anndata"),
    "sopa": import_status("sopa"),
    "spatialdata": import_status("spatialdata"),
    "shapely": import_status("shapely"),
}

dependency_status


## Dry-Run Planning

Leave `RUN_DRY = True` while you validate config resolution, output paths, and chunk planning.


In [ ]:
dry_run_plan = stage_call("run_all(dry_run=True)", run_all, config, SLIDE_ID, dry_run=True)

if RUN_MULTISLIDE_NIMBUS:
    multislide_nimbus_dry_run = stage_call(
        "run_nimbus_multislide(dry_run=True)",
        run_nimbus_multislide,
        config,
        MULTISLIDE_SLIDE_IDS,
        chunk_indices=NIMBUS_CHUNK_INDICES,
        dry_run=True,
    )
else:
    multislide_nimbus_dry_run = {
        "status": "skipped",
        "reason": "Set RUN_MULTISLIDE_NIMBUS = True to preview combined-slide Nimbus execution.",
    }


## Run Individual Stages

Set `RUN_DRY = False` above when you are on the cluster and ready to execute a real step.

When `RUN_MULTISLIDE_NIMBUS = True`, the pre-Nimbus stages below run in a simple loop over `MULTISLIDE_SLIDE_IDS`, then Nimbus runs once across that combined slide set. SpatialData and QC remain per-slide.


In [ ]:
stage_slide_ids = MULTISLIDE_SLIDE_IDS if RUN_MULTISLIDE_NIMBUS else [SLIDE_ID]
setup_results = {}

In [ ]:
for target_slide_id in stage_slide_ids:
    target_slide = get_slide_config(config, target_slide_id)
    if target_slide.get("setup"):
        setup_results[target_slide_id] = stage_call(
            f"setup_slide({target_slide_id}, dry_run={RUN_DRY})",
            setup_slide,
            config,
            target_slide_id,
            force=FORCE,
            dry_run=RUN_DRY,
        )
    else:
        print(f"No setup block configured for {target_slide_id}.")

setup_result = setup_results.get(SLIDE_ID)

In [ ]:
merge_results = {}

for target_slide_id in stage_slide_ids:
    merge_results[target_slide_id] = stage_call(
        f"merge_slide_ometiffs({target_slide_id}, dry_run={RUN_DRY})",
        merge_slide_ometiffs,
        config,
        target_slide_id,
        force=FORCE,
        dry_run=RUN_DRY,
    )

merge_result = merge_results[SLIDE_ID]


In [ ]:
instanseg_results = {}

for target_slide_id in stage_slide_ids:
    instanseg_results[target_slide_id] = stage_call(
        f"run_instanseg({target_slide_id}, dry_run={RUN_DRY})",
        run_instanseg,
        config,
        target_slide_id,
        force=FORCE,
        dry_run=RUN_DRY,
    )

instanseg_result = instanseg_results[SLIDE_ID]


In [ ]:
if RUN_MULTISLIDE_NIMBUS:
    nimbus_result = stage_call(
        f"run_nimbus_multislide(dry_run={RUN_DRY})",
        run_nimbus_multislide,
        config,
        MULTISLIDE_SLIDE_IDS,
        chunk_indices=NIMBUS_CHUNK_INDICES,
        force=FORCE,
        dry_run=RUN_DRY,
    )
else:
    nimbus_result = stage_call(
        f"run_nimbus_chunked(dry_run={RUN_DRY})",
        run_nimbus_chunked,
        config,
        SLIDE_ID,
        force=FORCE,
        dry_run=RUN_DRY,
    )


In [ ]:
if RUN_MULTISLIDE_NIMBUS and RUN_NIMBUS_FINALIZE:
    nimbus_finalize_result = stage_call(
        f"finalize_nimbus_multislide(dry_run=False)",
        finalize_nimbus_multislide,
        config,
        MULTISLIDE_SLIDE_IDS,
    )
else:
    nimbus_finalize_result = {
        "status": "skipped",
        "reason": "Set RUN_NIMBUS_FINALIZE = True after all multislide chunk jobs have finished.",
    }

nimbus_finalize_result


In [ ]:
spatialdata_results = {}

for target_slide_id in stage_slide_ids:
    spatialdata_results[target_slide_id] = stage_call(
        f"build_spatialdata({target_slide_id}, dry_run={RUN_DRY})",
        build_spatialdata,
        config,
        target_slide_id,
        force=FORCE,
        dry_run=RUN_DRY,
        return_sdata=False,
    )

spatialdata_result = spatialdata_results[SLIDE_ID]


In [ ]:
slide

In [ ]:
import spatialdata as sd

sdata = sd.read_zarr(slide["spatialdata"]["store_path"])
sdata


In [ ]:
diagnose_label_overlap_instances(sdata["cell_labels"], sdata["nuclear_labels"])


In [ ]:
config["slides"]


In [ ]:
import numpy as np
import tifffile
from pathlib import Path

cell_mask_path = Path(
    instanseg_result["cell_mask_path"]
    if "instanseg_result" in globals() and "cell_mask_path" in instanseg_result
    else Path(slide["mask_export"]["mask_dir"]) / f"{SLIDE_ID}{slide['mask_export']['suffix']}"
)
nuclear_mask_path = Path(
    instanseg_result["nuclear_mask_path"]
    if "instanseg_result" in globals() and "nuclear_mask_path" in instanseg_result
    else Path(slide["mask_export"]["mask_dir"]) / f"{SLIDE_ID}{slide['mask_export']['nuclear_suffix']}"
)

cells = np.asarray(tifffile.imread(cell_mask_path))
nuclear = np.asarray(tifffile.imread(nuclear_mask_path))

diagnose_label_overlap_instances(cells, nuclear)


In [ ]:
qc_result = stage_call("qc_slide", qc_slide, config, SLIDE_ID)


## Full Run Shortcut

After debugging individual stages, you can use the single `run_all(...)` wrapper below for one slide.

`run_all(...)` is still single-slide, so multislide Nimbus should be tested with the staged cells above rather than this shortcut.


In [ ]:
run_all_result = stage_call(
    f"run_all(dry_run={RUN_DRY})",
    run_all,
    config,
    SLIDE_ID,
    force=FORCE,
    dry_run=RUN_DRY,
)
